import wget

wget.download("https://raw.githubusercontent.com/AlexeyAB/darknet/master/cfg/yolov4.cfg")

wget.download("https://github.com/AlexeyAB/darknet/releases/download/yolov4/yolov4.weights")


wget.download("https://raw.githubusercontent.com/AlexeyAB/darknet/master/cfg/coco.data")

In [1]:
!PWD

'PWD' is not recognized as an internal or external command,
operable program or batch file.


# **yolo config ,weights, names**

In [2]:
import os
import numpy as np
import cv2 as cv
import imutils

In [3]:
conf = 0.5
threshold = 0.3

In [4]:
dir = "YOLO_V4"

In [5]:
label_path = os.path.join(dir,'coco.names')

In [6]:
Labels = open(label_path).read().strip().split("\n")

In [7]:
Labels

['person',
 'bicycle',
 'car',
 'motorbike',
 'aeroplane',
 'bus',
 'train',
 'truck',
 'boat',
 'traffic light',
 'fire hydrant',
 'stop sign',
 'parking meter',
 'bench',
 'bird',
 'cat',
 'dog',
 'horse',
 'sheep',
 'cow',
 'elephant',
 'bear',
 'zebra',
 'giraffe',
 'backpack',
 'umbrella',
 'handbag',
 'tie',
 'suitcase',
 'frisbee',
 'skis',
 'snowboard',
 'sports ball',
 'kite',
 'baseball bat',
 'baseball glove',
 'skateboard',
 'surfboard',
 'tennis racket',
 'bottle',
 'wine glass',
 'cup',
 'fork',
 'knife',
 'spoon',
 'bowl',
 'banana',
 'apple',
 'sandwich',
 'orange',
 'broccoli',
 'carrot',
 'hot dog',
 'pizza',
 'donut',
 'cake',
 'chair',
 'sofa',
 'pottedplant',
 'bed',
 'diningtable',
 'toilet',
 'tvmonitor',
 'laptop',
 'mouse',
 'remote',
 'keyboard',
 'cell phone',
 'microwave',
 'oven',
 'toaster',
 'sink',
 'refrigerator',
 'book',
 'clock',
 'vase',
 'scissors',
 'teddy bear',
 'hair drier',
 'toothbrush']

In [8]:
config_file_path = os.path.join(dir,'yolov4.cfg')
weights_path  = os.path.join(dir,'yolov4.weights')

In [9]:
np.random.seed(42)

COLORS = np.random.randint(0,255,(len(Labels),3),dtype='uint8')

In [10]:
net = cv.dnn.readNetFromDarknet(config_file_path,weights_path)

In [11]:
layer = net.getLayerNames()

In [12]:
print(layer)

('conv_0', 'bn_0', 'mish_1', 'conv_1', 'bn_1', 'mish_2', 'conv_2', 'bn_2', 'mish_3', 'identity_3', 'conv_4', 'bn_4', 'mish_5', 'conv_5', 'bn_5', 'mish_6', 'conv_6', 'bn_6', 'mish_7', 'shortcut_7', 'conv_8', 'bn_8', 'mish_9', 'concat_9', 'conv_10', 'bn_10', 'mish_11', 'conv_11', 'bn_11', 'mish_12', 'conv_12', 'bn_12', 'mish_13', 'identity_13', 'conv_14', 'bn_14', 'mish_15', 'conv_15', 'bn_15', 'mish_16', 'conv_16', 'bn_16', 'mish_17', 'shortcut_17', 'conv_18', 'bn_18', 'mish_19', 'conv_19', 'bn_19', 'mish_20', 'shortcut_20', 'conv_21', 'bn_21', 'mish_22', 'concat_22', 'conv_23', 'bn_23', 'mish_24', 'conv_24', 'bn_24', 'mish_25', 'conv_25', 'bn_25', 'mish_26', 'identity_26', 'conv_27', 'bn_27', 'mish_28', 'conv_28', 'bn_28', 'mish_29', 'conv_29', 'bn_29', 'mish_30', 'shortcut_30', 'conv_31', 'bn_31', 'mish_32', 'conv_32', 'bn_32', 'mish_33', 'shortcut_33', 'conv_34', 'bn_34', 'mish_35', 'conv_35', 'bn_35', 'mish_36', 'shortcut_36', 'conv_37', 'bn_37', 'mish_38', 'conv_38', 'bn_38', 'mish

In [13]:
layer = [layer [i-1] for i in net.getUnconnectedOutLayers()]

In [14]:
layer

['yolo_139', 'yolo_150', 'yolo_161']

# **Start frame**

In [15]:
cam = cv.VideoCapture("schoolzone.mp4")
H = None
W = None


while True:
    
    _,frame = cam.read()
    frame =  imutils.resize(image=frame,width=500)
    
    if W is None or H is None:
        (H,W) = frame.shape[:2]
        
    blob = cv.dnn.blobFromImage(frame,scalefactor=1/255.0,size=(416,416),swapRB=True,crop=False)
    
    net.setInput(blob)
    
    detections = net.forward(layer)
    
    boxes = []
    confidence = []
    class_idxs = []
    
    for output in detections:
        
        for detection in output:
            
            score = detection[5:]
            
            index = np.argmax(score)
            
            confidences = score[index]
            
            if confidences > conf:
                
                box = detection[0:4] * np.array([W,H,W,H])
                
                (center_x,center_y,width,height) = box.astype('int')
                
                x = int(center_x - (width//2))
                y = int(center_y - (height//2))
                
                boxes.append((x,y,int(width),int(height)))
                confidence.append(float(confidences))
                class_idxs.append(index)
                
                
                
                
    idxs = cv.dnn.NMSBoxes(boxes,confidence,conf,threshold)
    
    if len(idxs) > 0:
        
        for i in idxs.flatten():
            (x,y) = (boxes[i][0],boxes[i][1])
            (w,h) = (boxes[i][2],boxes[i][3])
            
            color = [int(c) for c in COLORS[class_idxs[i]]]
            cv.rectangle(frame,(x,y),(x+w,y+h),color,2)
            
            label = '{}: {:.4f}%'.format(Labels[class_idxs[i]],confidence[i])
            
            cv.putText(frame,label,(x,y-5),cv.FONT_HERSHEY_SIMPLEX,0.5,color,2)
            
    cv.imshow('frame',frame)
    
    if cv.waitKey(1) & 0xff == ord('r'):
        break
    
cam.release()
cv.destroyAllWindows()
                
                
                
    
    